# Regressionsbäume und Ensembles – Konzepte

Lesen Sie zuerst die Idee, dann die Formel und anschliessend das kleine Beispiel. Der Demonstrationscode ist bereits vorhanden.

## Bibliotheken und Einstellungen

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance

## 1. Ein Regressionsbaum sagt einen Mittelwert voraus

Ein Baum stellt Ja/Nein-Fragen wie „Hubraum ≤ 2 Liter?“. Jede Antwort führt zu einem anderen Knoten. Der letzte Knoten ist ein **Blatt**. Seine Vorhersage entspricht dem Mittelwert der Zielwerte der Trainingszeilen in diesem Blatt. Ein Baum unterteilt den Eingaberaum in Regionen, wobei jede Region eine konstante Vorhersage besitzt.

Um eine Teilung auszuwählen, berechnen wir die quadratischen Fehler um den Mittelwert jedes Teilbaums:
$$\mathrm{SSE}_{\mathrm{split}}=\sum_{i\in\mathrm{left}}(y_i-\bar y_{\mathrm{left}})^2+\sum_{i\in\mathrm{right}}(y_i-\bar y_{\mathrm{right}})^2.$$
Wir testen mögliche Merkmale und Schwellenwerte und wählen den kleinsten Gesamt-SSE. Diesen Vorgang wiederholen wir in jedem Unterbaum (**rekursive**, **greedy**-Teilung). Für die Zielwerte 10, 12, 14, 30 und 34 liefert eine Teilung nach 14 die Vorhersagen 12 und 32 sowie einen Gesamt-SSE von 16, verglichen mit 496 vor der Teilung.

**SSE** steht für *Sum of Squared Errors*, auf Deutsch ungefähr: Summe der quadrierten Fehler.

$$\mathrm{SSE}_{\mathrm{split}}=\sum_{i\in\mathrm{left}}(y_i-\bar y_{\mathrm{left}})^2+\sum_{i\in\mathrm{right}}(y_i-\bar y_{\mathrm{right}})^2.$$

Was sagt ein Blatt voraus?

Ein Blatt gibt immer den Durchschnitt der Zielwerte in diesem Blatt aus.

Generell:

- `x` ist das Merkmal, mit dem wir vorhersagen.
- `y` ist der Zielwert, den wir vorhersagen möchten.

In [5]:
y = np.array([10, 12, 14, 30, 34]) # Synthethische y-Werte
left = y[:3] # Linkes Blatt enthält die ersten 3 Werte von der Liste
right = y[3:] # Rechtes Blatt enthält die letzten 2 Werte von der Liste
print('Parent SSE:', np.sum((y - y.mean()) ** 2)) # beschreibt den quadrierten Fehler ohne Teilung
print('Leaf means:', left.mean(), right.mean())
print('Split SSE:', np.sum((left - left.mean()) ** 2) + np.sum((right - right.mean()) ** 2)) # beschreibt den quadrierten Fehler mit Teilung

Parent SSE: 496.0
Leaf means: 12.0 32.0
Split SSE: 16.0


## 2. Grösse, Hyperparameter und Beschneidung

Tiefe Bäume können Rauschen lernen. Ein **Hyperparameter** ist eine Einstellung, die wir vor dem Trainieren festlegen.

| Parameter | Bedeutung | Auswirkung einer Erhöhung |
|---|---|---|
| `max_depth` | maximale Tiefe | ermöglicht einen flexibleren Baum |
| `min_samples_split` | benötigte Zeilen, um einen Knoten zu teilen | verhindert kleine Teilungen |
| `min_samples_leaf` | Mindestanzahl an Zeilen pro Blatt | glättet die Vorhersagen |
| `max_leaf_nodes` | maximale Anzahl an Blättern | ermöglicht mehr Regionen |
| `max_features` | bei einer Teilung berücksichtigte Merkmale | mehr berücksichtigte Merkmale bedeuten weniger Zufälligkeit |
| `ccp_alpha` | Beschneidungsstrafe | führt zu einem kleineren Baum |
| `criterion` | Teilungskriterium | Regression: quadratischer Fehler; Klassifikation: Gini-Unreinheit/Entropie |

**Beschneidung (Pruning)** entfernt Äste, deren Nutzen im Verhältnis zu einer Strafe pro Blatt zu klein ist:
$$\mathrm{Beschneidungsziel}=\mathrm{Baumfehler}+\alpha\times\mathrm{Anzahl\ der\ Blätter}.$$
Dabei ist $\alpha$ die Strafe. `ccp_alpha=0` Es gibt keine Strafe für zusätzliche Blätter; grössere Werte können Äste entfernen. Scikit-learn gibt den Regressionsfehler pro Beobachtung an, daher hängen die Alpha-Werte von den Einheiten des Zielwerts ab. 
Der Baumfehler hängt von der Grösse der Zielwerte ab.

Beispiel:

Bei Preisen zwischen 10 und 50 sind Fehler relativ klein.
Bei Preisen zwischen 10'000 und 50'000 sind die quadrierten Fehler viel grösser.

Wählen Sie die Einstellungen mithilfe einer Kreuzvalidierung (CV) auf den Trainingsdaten.
```python
ccp_alpha = 0
ccp_alpha = 0.001
ccp_alpha = 0.01
ccp_alpha = 0.1
```

**Merksatz**

Entferne komplizierte Äste, wenn sie den Fehler nur wenig verbessern.

- kleines ccp_alpha → grosser, komplexerer Baum
- grosses ccp_alpha → kleiner, einfacherer Baum
- zu klein → Gefahr von Overfitting
- zu gross → Gefahr von Underfitting
Das Ziel ist ein Baum, der nicht nur die Trainingsdaten gut kennt, sondern auch bei neuen Daten gute Vorhersagen macht.

## 3. Bagging und Out-of-Bag-Zeilen

```text
Bei jeder Teilung:

Alle Merkmale prüfen
        ↓
Bestes Merkmal auswählen
```

**Bagging** trainiert Bäume auf Bootstrap-Stichproben (zufällig zusammengestellte neue Stichprobe aus den vorhandenen Trainingsdaten): Trainingszeilen werden **mit Zurücklegen** gezogen, **sodass einige Trainingszeilen mehrfach vorkommen und andere fehlen**. Die Vorhersagen werden gemittelt:
$$\hat y_{\mathrm{bag}}=\frac1B\sum_{b=1}^{B}\hat y_b.$$
$B$ ist die Anzahl der Bäume. Die Vorhersagen 8, 10 und 9 ergeben einen Durchschnitt von 9.

Bei einer grossen Stichprobe bleiben ungefähr 37 % der Zeilen in der Bootstrap-Stichprobe eines bestimmten Baums unberücksichtigt. Diese **Out-of-Bag-(OOB-)Zeilen** ermöglichen eine interne Bewertung mit Bäumen, die die jeweilige Zeile nicht gesehen haben. Die OOB-Bewertung ersetzt nicht den abschliessenden Test.

Das Mitteln reduziert die Instabilität. Bei gleicher Baumvarianz $\sigma^2$ und einer gemeinsamen Korrelation $\rho$ gilt:
$$\mathrm{Var}(\mathrm{average})=\rho\sigma^2+\frac{(1-\rho)\sigma^2}{B}.$$
Nur der unabhängige Anteil wird kleiner, wenn wir weitere Bäume hinzufügen. Für $\sigma^2=1$, $\rho=0.2$ und $B=100$ beträgt die Varianz 0.208. Eine Herleitung ist nicht erforderlich.

## 4. Random Forest (bagging + random features)

```text
Bei jeder Teilung:

Zufällige Merkmalsgruppe auswählen
        ↓
Nur diese Merkmale prüfen
        ↓
Bestes Merkmal aus dieser Gruppe auswählen
```

Ein Random Forest ergänzt das Bagging durch **zufällige Merkmalsuntergruppen bei jeder Teilung**. Dadurch ähneln sich die Bäume weniger, sodass das Mitteln stärker helfen kann. Verwenden Sie `n_estimators` für die Anzahl der Bäume und `max_features` für die Merkmale, die bei jeder Teilung berücksichtigt werden.

In scikit-learn verwendet ein Regressionswald standardmässig alle Merkmale (`max_features=1.0`); ein Klassifikationswald verwendet standardmässig `"sqrt"`. Mehr Bäume stabilisieren die Ergebnisse, erhöhen aber den Rechenaufwand, während der zusätzliche Nutzen abnimmt.

Die Merkmalswichtigkeit (welche Merkmale bei den Vorhersagen besonders hilfreich waren) zeigt, welche Merkmale zur Vorhersage beitragen. Die auf Unreinheit basierende Wichtigkeit kann einige Merkmale unfair bevorzugen. Bei der **Permutationswichtigkeit** wird ein Merkmal zufällig durchmischt und der Rückgang der Vorhersageleistung gemessen. Wichtigkeit ist kein kausaler Effekt.

## 5. Gradient Boosting und Vergleich

Boosting erstellt kleine Bäume **nacheinander**. Beim quadratischen Verlust lernt jeder neue Baum die aktuellen Residuen:
$$r_i=y_i-F_{m-1}(x_i),\qquad F_m(x)=F_{m-1}(x)+\nu h_m(x).$$
$F_{m-1}$ ist das aktuelle Modell, $h_m$ der nächste Baum und $\nu$ die Lernrate. Wenn die aktuelle Vorhersage 7 beträgt, der nächste Baum ein Residuum von 3 vorhersagt und $\nu=0.1$ gilt, lautet die aktualisierte Vorhersage $7+0.1\times3=7.3$. Andere differenzierbare Verlustfunktionen verwenden negative Gradienten (Pseudo-Residuen).

**Kann systematische Fehler, Bias und Varianz reduzieren**

---

| Modell | Hauptidee | Wichtige Einstellungen |
|---|---|---|
| Lineare Regression | gewichtete Summe der Merkmale | ausgewählte Merkmale |
| Einzelner Baum | verständliche Regeln und konstante Blattvorhersagen | Tiefe, Beschneidung |
| Random Forest | Mittelwert vieler unterschiedlicher Bäume | Bäume, zufällige Merkmale |
| Boosting | schrittweise Korrektur der Fehler | Lernrate, Bäume, Tiefe |

- Bagging reduziert hauptsächlich die Varianz; 
- Boosting verringert schrittweise den systematischen Fehler und kann sowohl Bias als auch Varianz beeinflussen. 

Vergleichen Sie die Validierungsergebnisse: Keine Methode ist immer überlegen.

In [6]:
old_prediction = 7
residual_prediction = 3
learning_rate = 0.1
print('Updated prediction:', old_prediction + learning_rate * residual_prediction)

Updated prediction: 7.3


### Jupyter-Notebook – Informationen für die Abgabe

Fügen Sie diese Informationen bei der Abgabe Ihres Notebooks ein.

In [7]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))

Linux-7.0.14-linuxkit-aarch64-with-glibc2.41
Python: 3.11.17
Date: 2026-10-27T13:07:01
